# Notebook 08 : Machine Learning sur les attributs EEG

Ce notebook guide la construction de pipelines de Machine Learning (ML) pour classer des essais EEG (ex: Go vs NoGo, Congruent vs Incongruent, etc.).

Nous partons des fichiers CSV dérivés des notebooks d'extraction d'attributs. Nous construirons et évaluerons plusieurs classificateurs, incluant la Régression Logistique (avec régularisation L1/L2), les Random Forests (Forêts d'arbres décisionnels) et les SVMs.

L'évaluation se concentrera sur une validation croisée stratifiée, l'accuracy (précision), l'accuracy équilibrée (balanced accuracy), les matrices de confusion, et l'importance des attributs.

### Remarque importante

Ce notebook reprend la structure `Type 1 / Type 2 / Type 3` introduite dans les modules précédents :
- **Type 1 — prêt à exécuter** : cellules complètes, sans modification nécessaire, pour établir une baseline.
- **Type 2 — à personnaliser** : blocs contenant des paramètres à ajuster ou des analyses à compléter.
- **Type 3 — exploration libre** : propositions d'analyses avancées (ex: tests de permutation, validation croisée LOSO).

> **Note technique (scikit-learn) :** Un modèle doit être ré-instancié (ex: `mon_modele = LogisticRegression(...)`) avant *chaque* nouvel entraînement (`.fit()` ou `cross_validate`). Les modèles ne se réinitialisent pas automatiquement. Pour garantir que nos expériences (ERP vs Fréquence vs Fusion) sont indépendantes, nous allons **toujours créer une nouvelle instance du modèle** avant de l'évaluer.

## Pour bien démarrer
- **Assurez-vous d'avoir exécuté les pipelines d'extraction (ERP et Fréquence)** pour générer les fichiers CSV finaux (ex: `all_subjects_epoch_erp_features.csv`, `all_subjects_epoch_band_features.csv`).
- **Vous devrez spécifier** le chemin vers vos fichiers CSV et la ou les colonnes cibles (target) à prédire (ex: 'condition').
- Activez l'environnement Python du cours et installez les dépendances (ex: `scikit-learn`, `pandas`, `matplotlib`).

## Objectifs pédagogiques
1. Charger les fichiers CSV d'attributs (features) et les préparer pour le ML (nettoyage, encodage des cibles 'y').
2. Entraîner et évaluer un modèle de **baseline** en utilisant uniquement les attributs de **fréquence** (`..._epoch_band_features.csv`) et générer une **matrice de confusion**.
3. Entraîner et évaluer un second modèle en utilisant uniquement les attributs **ERP** (`..._epoch_erp_features.csv`) et générer une **matrice de confusion**.
4. **Fusionner** les deux jeux d'attributs (ERP + Fréquence) pour construire un modèle complet et évaluer si la combinaison améliore la performance.
5. Explorer des méthodes d'évaluation avancées : validation croisée de type **LOSO (Leave-One-Subject-Out)**, visualisation de **l'importance des attributs** (feature importance), et exécution d'un **test de permutation** pour valider la significativité du modèle.

# 0. Configuration et fonctions utilitaires
Nous conservons un bloc d'import unique, définissons les chemins d'accès aux données, et écrivons des fonctions utilitaires réutilisables.
Cela permet aux trois blocs d'analyse (Type 1, 2 et 3) de rester concentrés sur les décisions de modélisation.

In [ ]:
import pandas as pd  # pandas pour la manipulation de tableaux (DataFrame)
import numpy as np  # numpy pour les opérations numériques et tableaux
from pathlib import Path  # Path pour construire et manipuler des chemins de fichiers
import matplotlib.pyplot as plt  # matplotlib pour créer des graphiques
import seaborn as sns  # seaborn pour améliorer l'esthétique des graphiques

from sklearn.preprocessing import StandardScaler, LabelEncoder  # StandardScaler pour normaliser, LabelEncoder pour encoder les labels
from sklearn.pipeline import Pipeline  # Pipeline pour chaîner prétraitement et modèle
from sklearn.linear_model import LogisticRegression  # régression logistique comme classifieur linéaire
from sklearn.ensemble import RandomForestClassifier  # forêt aléatoire pour importance des features et classification
from sklearn.svm import SVC  # SVM pour classification (noyau RBF possible)
from sklearn.model_selection import StratifiedGroupKFold, GridSearchCV, cross_validate, cross_val_predict, LeaveOneGroupOut, permutation_test_score  # outils de validation croisée et recherche d'hyperparam
from sklearn.metrics import ConfusionMatrixDisplay  # utilitaire pour afficher les matrices de confusion
from sklearn.inspection import permutation_importance  # importance des features par permutation

sns.set_context('talk')  # règle le contexte de taille de police pour les figures (utile pour notebooks de présentation)
sns.set_style('whitegrid')  # applique un style de fond avec grille blanche pour les graphiques

RANDOM_STATE = 42  # valeur de graine aléatoire pour garantir la reproductibilité
ROOT = Path('tasks/TASKFOLDER/bids/derivatives/TASKNAME-analysis')  # chemin racine vers le dossier de données dérivées
paths = {
    'epoch_band': ROOT / 'all_subjects_epoch_band_features.csv',  # chemin vers le CSV des features de bande au niveau epoch
    'epoch_erp': ROOT / 'all_subjects_epoch_erp_features.csv',    # chemin vers le CSV des features ERP au niveau epoch
    'evoked_band': ROOT / 'all_subjects_evoked_band_features.csv',# chemin vers le CSV des features de bande pour evoked (moyennes)
    'evoked_erp': ROOT / 'all_subjects_evoked_erp_features.csv',  # chemin vers le CSV des features ERP pour evoked (moyennes)
}

pd.options.display.max_columns = 200  # configure pandas pour afficher jusqu'à 200 colonnes dans la repr

for name, path in paths.items():  # itère sur le dictionnaire des chemins
    print(f"- {name:12} -> {path}")  # affiche chaque clé et son chemin associé pour vérification

# 1. ML avec les bandes de fréquences au niveau des essai

nous allons commencer par les attributs (features) extraits au niveau de l'essai (`epoch-level`). Ce sont les fichiers CSV où chaque ligne représente un essai individuel (ex: `all_subjects_epoch_band_features.csv`).

Notre objectif : entraîner un modèle de Machine Learning à prédire la condition d'un essai unique.

Les autres fichiers que nous avons générés (ex: `all_subjects_evoked_features.csv`) représentent les moyennes par condition (`Evoked`), vous pourrez les utiliser plus tard pour les mêmes analyses.

Chaque bloc de ce notebook chargera le fichier CSV cible (ERP, Fréquence, ou les deux fusionnés) via les utilitaires de configuration définis ci-dessous.

### Bloc de Type 1 : Constantes et fonctions utilitaires

Ici, nous définissons le **socle technique** commun à toutes les analyses de ce notebook (Type 1, 2 et 3). L'objectif est de centraliser la configuration et la logique réutilisable pour éviter la duplication de code et garantir la cohérence de nos expériences.

Ce bloc contient :

1.  **Les configurations des modèles** (`CLASSIFIER_CONFIG`) : Une structure pour définir les hyperparamètres de base des classificateurs que nous allons tester (Régression Logistique, SVM, etc.).
2.  **La constante de reproductibilité** (`RANDOM_SEED`) : Une "graine aléatoire" fixe pour s'assurer que toutes les opérations stochastiques (comme la séparation des données) donnent les mêmes résultats à chaque exécution.
3.  **Les métriques d'évaluation** (`COMMON_SCORING`) : La liste des scores (ex: `accuracy`) que nous utiliserons pour juger les performances de nos modèles.
4.  **Les fonctions utilitaires (helpers)** : Des fonctions Python réutilisables pour :
    * `load_dataframe` : Charger nos fichiers CSV.
    * `prepare_dataset` : Transformer un DataFrame en matrices `X` (features), `y` (labels), et `groups` (sujets) prêtes pour `scikit-learn`.
    * `make_cv` : Créer notre stratégie de validation croisée (`StratifiedGroupKFold`) qui respecte la séparation des sujets.
    * `evaluate_models` : Une boucle d'évaluation complète qui entraîne et score nos modèles.
    * `combine_feature_sets` : Une fonction pour fusionner différents ensembles d'attributs (ex: ERP + Fréquence).

In [ ]:
from typing import Tuple  # pour les annotations de type

# Dictionnaire global pour configurer les classificateurs que nous allons tester.
# Note: 'class_weight': 'balanced' est crucial si nos classes sont déséquilibrées.
# Note2: 'penalty': 'l2' est la régularisation par défaut pour LogisticRegression avec 'lbfgs',
# mais nous le spécifions explicitement pour la clarté. Il est possible d'expérimenter avec 'l1' ou 'elasticnet'.
# 'max_iter': Augmenté pour assurer la convergence des solveurs.
CLASSIFIER_CONFIG = {
    "LogisticRegression": {
        "type": "logreg", 
        "params": {"class_weight": "balanced", "max_iter": 1000, "solver": "lbfgs", "penalty": "l2"}
    },
    "LinearSVM": {
        "type": "linear_svm", 
        "params": {"class_weight": "balanced", "max_iter": 5000}
    },
    "LDA": {
        "type": "lda", 
        "params": {} # Analyse Discriminante Linéaire, simple baseline.
    },
    "RandomForest": {
        "type": "random_forest", 
        "params": {"class_weight": "balanced_subsample", "n_estimators": 100, "max_depth": None, "random_state": RANDOM_STATE}
    },
}

# Graine aléatoire (seed) pour assurer la reproductibilité de nos résultats
# (ex: pour le 'shuffle' dans la validation croisée).
RANDOM_SEED = 42


# Liste des métriques de scoring que nous demanderons à cross_validate.
# 'balanced_accuracy' est particulièrement importante si les classes sont déséquilibrées.
COMMON_SCORING = ['accuracy', 'balanced_accuracy']

def load_dataframe(path: Path) -> pd.DataFrame:
    """Fonction utilitaire simple pour charger un fichier CSV."""
    if not path.exists():
        raise FileNotFoundError(f"Fichier CSV introuvable : {path}")
    return pd.read_csv(path)

def prepare_dataset(df: pd.DataFrame, drop_trial_index: bool = True) -> Tuple[pd.DataFrame, np.ndarray, np.ndarray, LabelEncoder]:
    """
    Prépare le DataFrame pour le ML.
    Sépare les 'features' (X), les 'labels' (y), et les 'groups' (sujets).
    
    Returns:
        Tuple: (X_numeric, y_encoded, groups, encoder)
    """
    # Définir les colonnes qui ne sont PAS des features (attributs).
    to_drop = ['subject', 'condition', 'session', 'run']
    if drop_trial_index:
        to_drop.append('trial_index')
    
    # 1. Créer X (les features)
    # 'errors='ignore'' évite un crash si une colonne (ex: 'trial_index') n'existe pas.
    features = df.drop(columns=to_drop, errors='ignore')
    # S'assurer que X ne contient que des colonnes numériques.
    numeric = features.select_dtypes(include=[np.number]).copy()
    
    # 2. Créer y (la cible) — regrouper "correct/incorrect" sous "stimulus/go" ou "stimulus/nogo"
    # Normalise la colonne 'condition' en ne gardant que les deux premiers segments (ex: 'stimulus/go').
    cond_simple = df['condition'].astype(str).str.split('/', expand=True).iloc[:, :2].apply(lambda row: '/'.join(row.dropna()), axis=1)
    # Vérification de la présence des conditions valides
    # Filtrer pour ne garder que les trials 'stimulus/go' et 'stimulus/nogo'
    valid = cond_simple.isin(['stimulus/go', 'stimulus/nogo'])
    if valid.any():
        # Appliquer le filtre aux données pour maintenir l'alignement entre X, y et groups
        df = df.loc[valid].reset_index(drop=True)
        numeric = numeric.loc[valid].reset_index(drop=True)
        cond_simple = cond_simple.loc[valid].reset_index(drop=True)
    # Encoder les étiquettes réduites
    encoder = LabelEncoder()
    y = encoder.fit_transform(cond_simple)
    
    # 3. Créer les groupes (pour la validation croisée par sujet)
    # C'est crucial pour que les données d'un même sujet ne se retrouvent pas
    # à la fois dans l'ensemble de train et de test (fuite de données).
    groups = df['subject'].astype(str)
    
    # Retourne les 3 composantes + l'encodeur (utile pour décoder les prédictions plus tard)
    return numeric, y, groups, encoder

def make_cv(n_splits=4) -> StratifiedGroupKFold:
    """Crée notre objet de validation croisée (Cross-Validator)."""
    # Nous utilisons StratifiedGroupKFold pour :
    # 1. 'Group': Garder les sujets (groups) séparés.
    # 2. 'Stratified': Maintenir l'équilibre des classes (y) dans chaque fold.
    # 3. 'shuffle=True': Mélanger les groupes (sujets) avant de les séparer.
    return StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_SEED)

def evaluate_models(models: dict, X: pd.DataFrame, y: np.ndarray, groups: np.ndarray, cv=None, scoring=None) -> pd.DataFrame:
    """
    Évalue une liste de modèles en utilisant la validation croisée groupée.
    
    Args:
        models (dict): Un dictionnaire de modèles scikit-learn (ex: {'LDA': lda_model}).
        X, y, groups: Les sorties de 'prepare_dataset'.
    
    Returns:
        pd.DataFrame: Un tableau résumé des scores pour chaque modèle.
    """
    # Utiliser nos valeurs par défaut si rien n'est fourni
    cv = cv or make_cv()
    scoring = scoring or COMMON_SCORING
    
    summary = [] # Pour stocker les résultats
    
    print(f"Évaluation de {len(models)} modèles en utilisant {cv.get_n_splits()}-fold CV...")
    
    # Boucler sur chaque modèle fourni
    for name, model in models.items():
        # Lancer la validation croisée
        # 'groups=groups' est le paramètre clé pour la séparation par sujet.
        # 'n_jobs=-1' utilise tous les cœurs du CPU pour accélérer.
        scores = cross_validate(model, X, y, groups=groups, cv=cv, scoring=scoring, n_jobs=-1)
        
        # Formater les résultats
        row = {'model': name}
        for metric in scoring:
            # Calculer la moyenne des scores obtenus sur les 'n_splits' folds
            mean_score = float(np.mean(scores[f'test_{metric}']))
            row[f'{metric}_mean'] = mean_score
            print(f"  -> {name} | {metric}_mean = {mean_score:.3f}")
            
        summary.append(row)
        
    # Retourner un DataFrame pandas propre, arrondi à 3 décimales
    return pd.DataFrame(summary).round(3)

def combine_feature_sets(primary: pd.DataFrame, secondary: pd.DataFrame) -> pd.DataFrame:
    """
    Fusionne deux ensembles de features (ex: ERPs + TFR).
    Prend toutes les features de 'primary' et ajoute les features numériques de 'secondary'.
    """
    # Garde-fou : s'assurer que les deux tables ont le même nombre d'essais
    if len(primary) != len(secondary):
        raise ValueError(f"Les tables n'ont pas le même nombre de lignes : {len(primary)} != {len(secondary)}")
    
    # 1. Isoler les features numériques de la table secondaire
    # (On ne veut pas dupliquer les colonnes 'subject' ou 'condition')
    secondary_numeric = secondary.drop(columns=['subject', 'condition', 'trial_index'], errors='ignore').select_dtypes(include=[np.number])
    
    # 2. Concaténer les deux tables côte à côte (axis=1)
    # .reset_index(drop=True) est une sécurité pour éviter les problèmes d'alignement d'index.
    combined = pd.concat([primary.reset_index(drop=True), secondary_numeric.reset_index(drop=True)], axis=1)
    
    return combined

print("Bloc 1 (Utilitaires) chargé avec succès.")

#### Bloc de Type 1 — Modèle de baseline sur les attributs de fréquence (epochs)

Nous travaillons avec le fichier `all_subjects_epoch_band_features.csv`. Nous définissons la cible (`y`) comme étant la colonne `condition` et nous évaluons trois classificateurs : une régression logistique, une forêt aléatoire simple ("shallow random forest"), et un SVM.

Toute la validation croisée (CV) est stratifiée par `condition` et groupée par `subject`.

In [ ]:
print("--- Lancement du Bloc 1 : Baseline sur les Bandes de Fréquence ---")

# --- 1. Chargement des données (Attributs de Fréquence) ---

# Charger le DataFrame en utilisant notre fonction utilitaire.
# 'paths' est supposé être un dictionnaire défini dans le bloc de configuration 0.
band_df = load_dataframe(paths['epoch_band'])

print(f"DataFrame des bandes de fréquence chargé : {band_df.shape[0]} essais x {band_df.shape[1]} colonnes")

# Nous utilisons print() pour afficher l'aperçu dans la console.
print("\nAperçu des données (bandes de fréquence) :")
print(band_df.head())

# --- 2. Préparation pour le ML ---

# Utiliser notre fonction utilitaire pour séparer les données en :
# X_band : Matrice d'attributs (features) numériques
# y_band : Vecteur cible (labels) encodé (ex: 0, 1)
# groups_band : Liste des ID de sujet pour la validation croisée
# band_encoder : L'objet LabelEncoder (pour retrouver les noms des classes)
X_band, y_band, groups_band, band_encoder = prepare_dataset(band_df)

# --- 3. Vérification (Logging) ---

# Afficher les classes (conditions) que l'encodeur a trouvées
# ex: ['stimulus/go/correct', 'stimulus/nogo/correct', ...]
print(f"\nConditions (cibles) trouvées : {band_encoder.classes_}")

# Afficher la forme de notre matrice X prête pour l'entraînement
print(f"Forme de la matrice X (features) : {X_band.shape}")
print(f"Nombre d'essais (y) : {y_band.shape[0]}")
print(f"Nombre de groupes (sujets) : {np.unique(groups_band).size}")

# AMÉLIORATION : Remplacement du message de log générique
print("\n-> Préparation des données (Bandes de Fréquence) terminée.")

#### Bloc de Type 1 — Définition des modèles (Pipelines)

Avant d'entraîner, nous devons définir nos modèles. Il est crucial d'inclure un `StandardScaler` dans un `Pipeline`, car la Régression Logistique et les SVMs y sont très sensibles. Le Random Forest n'en a pas *besoin*, mais l'inclure ne pose pas de problème et uniformise le processus.

In [ ]:
print("--- Définition des pipelines de classification ---")

# 1. Pipeline pour la Régression Logistique
# Nous utilisons les paramètres de notre 'CLASSIFIER_CONFIG'
pipe_lr = Pipeline([
    ('scaler', StandardScaler()), # Étape 1 : Normaliser les données
    ('model', LogisticRegression(
        **CLASSIFIER_CONFIG['LogisticRegression']['params'],
        random_state=RANDOM_SEED
    ))
])


# Dictionnaire de tous les modèles à évaluer
models = {
    'LogisticRegression': pipe_lr,
}

print(f"{len(models)} modèles (pipelines) ont été créés.")

#### Bloc de Type 1 — Évaluation des modèles (Scores)

Maintenant, nous utilisons notre fonction utilitaire `evaluate_models` pour entraîner tous les modèles définis ci-dessus sur les données `X_band` et obtenir les scores de performance.

In [ ]:
# 1. Créer notre stratégie de validation croisée (CV)
# Utilise la fonction 'make_cv()' définie dans le bloc utilitaire
cv = make_cv()

# 2. Évaluer les modèles
# 'evaluate_models' s'occupe de tout :
# - Boucle sur chaque modèle
# - Exécute 'cross_validate'
# - Respecte les groupes (sujets)
# - Calcule la moyenne des scores
scores_df = evaluate_models(
    models, 
    X_band, 
    y_band, 
    groups_band, 
    cv=cv
)

# 3. Afficher les résultats
print("\n--- Scores de performance (Baseline : Bandes de Fréquence) ---")
print(scores_df)

#### Bloc de Type 1 — Matrice de Confusion (Random Forest)

Les scores de performance (comme `balanced_accuracy`) nous donnent un chiffre global, mais une matrice de confusion nous montre *où* le modèle se trompe.

Nous allons en générer une en utilisant `cross_val_predict`. Cette fonction entraîne le modèle en validation croisée et retourne les prédictions faites sur les ensembles de test de chaque fold. C'est la manière la plus juste de générer des prédictions sur l'ensemble du jeu de données pour visualiser la performance.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

print("\n--- Calcul de la Matrice de Confusion pour le Random Forest ---")

# 1. Sélectionner le modèle que nous voulons visualiser
model_to_plot = models['LogisticRegression']

# 2. Obtenir les prédictions pour chaque essai (epoch)
# 'cross_val_predict' exécute la CV et retourne les prédictions 'y_pred'
# pour l'ensemble de test de chaque fold. C'est la bonne façon de
# générer des prédictions sur l'ensemble du jeu de données.
y_pred = cross_val_predict(
    model_to_plot, 
    X_band, 
    y_band, 
    groups=groups_band, 
    cv=cv, 
    n_jobs=-1
)

# 3. Calculer la matrice de confusion
# Compare les vraies étiquettes (y_band) aux prédictions (y_pred)
cm = confusion_matrix(y_band, y_pred)

# 4. Afficher la matrice
# Nous utilisons 'band_encoder.classes_' (de notre bloc de préparation)
# pour obtenir les noms lisibles (ex: 'stimulus/go/correct')
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm, 
    display_labels=band_encoder.classes_
)

print("Affichage de la matrice de confusion...")
disp.plot(cmap='Blues', xticks_rotation=45)
plt.title('Matrice de Confusion - LR (Attributs de Fréquence)')
plt.show()

#

#### Bloc de Type 2 — Analyse sur les attributs ERP

Nous allons maintenant répéter l'analyse de baseline (Type 1), mais en utilisant cette fois un jeu de données différent.

**Vos tâches :**
1.  Chargez le fichier `all_subjects_epoch_erp_features.csv` en utilisant la fonction `load_dataframe`.
2.  Préparez les données (`X_erp`, `y_erp`, `groups_erp`) en utilisant la fonction `prepare_dataset`.
3.  Ré-évaluez les modèles de base (le dictionnaire `models` du Bloc 1) sur ces nouvelles données `X_erp` en appelant `evaluate_models`.
4.  Calculez et affichez la matrice de confusion pour le meilleur modèle (tout comme vous l'avez fait dans le Bloc 1).
5.  Comparez les scores (ex: `balanced_accuracy`) obtenus avec les attributs ERP à ceux obtenus avec les attributs de fréquence.

### Une note pédagogique : Pourquoi les attributs ERP "single-trial" sont-ils bruités ?

Vous remarquerez probablement que les scores pour les attributs ERP sont plus bas que ceux des bandes de fréquence. C'est normal.

Les ERPs (comme la P3 ou la N2) sont des signaux très faibles, profondément enfouis dans le bruit de fond de l'EEG. Ils ne deviennent clairement visibles qu'**après avoir moyenné** des dizaines d'essais (ce que nous avons fait dans l'analyse `Evoked`). Sur un **essai unique (`Epoch`)**, le signal ERP est extrêmement bruité. L'amplitude ou la latence "pic" que nous extrayons peut souvent être juste un pic de bruit aléatoire.

En revanche, la **puissance de bande (Fréquence)** est une mesure moyennée sur une fenêtre de temps, ce qui la rend intrinsèquement plus stable et moins sensible au bruit instantané. C'est pourquoi les attributs de fréquence sont souvent de meilleurs prédicteurs pour le ML au niveau de l'essai.

#### Bloc de Type 3 — Modèles avancés

Dans ce bloc, nous allons monter en complexité pour voir si nous pouvons améliorer nos scores.

**Nouveaux modèles et métriques**
1.  Créez un nouveau dictionnaire `models_advanced`. Incluez-y le modèle `LogisticRegression_L2` (celui du Bloc 1), mais ajoutez-y aussi :
    * Un pipeline pour `LinearDiscriminantAnalysis` (LDA).
    * Un pipeline pour `LogisticRegression` avec une pénalité `penalty='l1'` (Lasso) et un `solver='liblinear'`.
    * Un pipeline pour `SVM`
    * Un pipeline pour `Random Forest`
    * Autre pipeline avec des modéles de votre choix.
2.  Créez un nouveau dictionnaire de `scoring_advanced` qui inclut `accuracy`, `balanced_accuracy`, mais aussi `f1_weighted` et `roc_auc_ovr` 
3.  Créez un nouvel objet CV `cv_5_splits` en appelant `make_cv(n_splits=5)`.
4.  Lancez `evaluate_models` en utilisant les données `X` et vos nouvelles configurations (`models_advanced`, `scoring_advanced`, `cv_5_splits`).
5.  Analysez les résultats. Quel modèle s'en sort le mieux ?

#### Bloc de Type 3 — Validation Croisée LOSO (Leave-One-Subject-Out)

`StratifiedGroupKFold` est rapide et robuste. Cependant, la méthode "gold standard" pour l'EEG est la **Leave-One-Subject-Out (LOSO)**. On entraîne le modèle sur tous les sujets *sauf un*, et on teste sur le sujet restant. On répète l'opération pour chaque sujet. C'est lent, mais cela prouve que le modèle généralise à de *nouveaux individus*.

**Vos tâches :**
1.  Importez `LeaveOneGroupOut` depuis `sklearn.model_selection`.
2.  Créez une instance de cet objet : `loso_cv = LeaveOneGroupOut()`.
3.  Appelez à nouveau `evaluate_models` en utilisant les données fusionnées (`X`, `y`, `groups`) et vos modèles avancés, mais cette fois-ci, passez `cv=loso_cv`.
4.  Comparez les scores LOSO aux scores `StratifiedGroupKFold`. Sont-ils plus bas ? C'est normal, car la tâche (prédire sur un sujet entièrement inconnu) est beaucoup plus difficile.

#### Bloc de Type 3 — Exploration : Qu'en est-il des données `Evoked` ?

Vous avez demandé si on pouvait entraîner des modèles sur les fichiers `..._evoked_features.csv`. La réponse est **oui**.

Vous pouvez refaire toutes les étapes mais avec evoked. 

## 2. Fusion des attributs (Bandes de Fréquence + ERP)

Nous avons évalué les attributs de fréquence (Bloc 1) et les attributs ERP séparément. Il est temps de les combiner pour voir si le modèle peut exploiter les deux types d'information (temporelle et fréquentielle) pour améliorer ses prédictions.

Dans ce bloc, nous allons :
1.  **Fusionner** les deux DataFrames (`band_df` et `erp_df`) en utilisant la fonction `combine_feature_sets`.
2.  **Préparer** ce nouveau jeu de données fusionné (`X_comb`, `y_comb`, `groups_comb`).
3.  **Lancer un pipeline d'évaluation simple** (similaire au Bloc 1, en utilisant `evaluate_models`) pour obtenir un score de performance pour ces données combinées.

In [ ]:
print("--- Lancement du Bloc 2 : Fusion des Attributs ---")

# --- 1. Chargement des jeux de données ---
print("Chargement des DataFrames (bandes de fréquence et ERP)...")

# Charger les attributs des bandes de fréquence (créés dans le notebook 07)
band_df = load_dataframe(paths['epoch_band'])
print(f"  -> DataFrame 'band_df' (fréquence) chargé. Forme: {band_df.shape}")

# Charger les attributs ERP (créés dans le notebook 06)
erp_df = load_dataframe(paths['epoch_erp'])
print(f"  -> DataFrame 'erp_df' (ERP) chargé. Forme: {erp_df.shape}")

# --- 2. Fusion des attributs ---
print("\nFusion des deux jeux de données...")

# Utiliser notre fonction utilitaire 'combine_feature_sets'
# 'primary' (band_df) garde ses colonnes non numériques (subject, condition)
# 'secondary' (erp_df) n'ajoute que ses colonnes numériques
combined_df = combine_feature_sets(band_df, erp_df)

# --- 3. Préparation du jeu de données fusionné ---
print("Préparation du jeu de données fusionné pour le ML...")

# Utiliser 'prepare_dataset' pour créer X (features), y (labels), et groups (sujets)
X_combined, y_combined, groups_combined, combined_encoder = prepare_dataset(combined_df)

# --- 4. Vérification et Affichage ---
print("\n--- Résumé du jeu de données fusionné ---")
print(f"Forme de la table combinée (brute) : {combined_df.shape}")
print(f"Nombre total d'attributs (features) pour le ML (X_combined) : {X_combined.shape[1]}")
print(f"Nombre total d'essais (y_combined) : {y_combined.shape[0]}")
print(f"Classes trouvées : {combined_encoder.classes_}")

print("\nAperçu de la table fusionnée (combined_df) :")
# Ajouter print() pour s'assurer que .head() s'affiche dans le script/log
print(combined_df.head())

In [ ]:
print("--- Définition des pipelines de classification ---")

# 1. Pipeline pour la Régression Logistique
# Nous utilisons les paramètres de notre 'CLASSIFIER_CONFIG'
pipe_lr = Pipeline([
    ('scaler', StandardScaler()), # Étape 1 : Normaliser les données
    ('model', LogisticRegression(
        **CLASSIFIER_CONFIG['LogisticRegression']['params'],
        random_state=RANDOM_SEED
    ))
])


# Dictionnaire de tous les modèles à évaluer
models = {
    'LogisticRegression': pipe_lr,
}

print(f"{len(models)} modèles (pipelines) ont été créés.")

In [ ]:
# 1. Créer notre stratégie de validation croisée (CV)
# Utilise la fonction 'make_cv()' définie dans le bloc utilitaire
cv = make_cv()

# 2. Évaluer les modèles
# 'evaluate_models' s'occupe de tout :
# - Boucle sur chaque modèle
# - Exécute 'cross_validate'
# - Respecte les groupes (sujets)
# - Calcule la moyenne des scores
scores_df = evaluate_models(
    models, 
    X_band, 
    y_band, 
    groups_band, 
    cv=cv
)

# 3. Afficher les résultats
print("\n--- Scores de performance (Baseline : Bandes de Fréquence) ---")
print(scores_df)

### Bloc de Type 3 (Optionel) — Importance des attributs (avec Régression Logistique)

Les scores de performance sont une chose, mais *quels* attributs sont les plus utiles ?

Une méthode très interprétable consiste à inspecter les **coefficients** d'un modèle linéaire, comme la **Régression Logistique**. Puisque nous avons normalisé nos données avec `StandardScaler`, nous pouvons directement comparer les coefficients :

* Un **coefficient élevé (positif ou négatif)** signifie que l'attribut a un **poids important** dans la décision.
* Le **signe** (+ ou -) nous dit *dans quelle direction* l'attribut pousse la prédiction (ex: vers 'Go' ou 'NoGo').

Nous utiliserons le modèle `LogisticRegression` avec `l2` (Ridge) mais `LogisticRegression` avec `l1` (Lasso) force les attributs inutiles à avoir un coefficient de **zéro**, ce qui en fait un excellent outil de **sélection d'attributs**. A tester. 

Ensuite, à titre de comparaison, nous utiliserons `permutation_importance` sur ce *même* modèle. Cette méthode ne regarde pas les coefficients, mais mesure à quel point le score chute lorsque nous mélangeons un attribut. C'est une excellente façon de valider nos résultats.

In [ ]:
from sklearn.inspection import permutation_importance
import matplotlib.pyplot as plt


# --- 1. Importance basée sur les Coefficients (Modèle L1) ---
print("  ... 1/2 Calcul des coefficients du modèle L1 (Lasso)...")

# Sélectionner le pipeline L1 (il doit être défini dans un bloc précédent)
model_l2_pipe = Pipeline([
    ('scaler', StandardScaler()), # Étape 1 : Normaliser les données
    ('model', LogisticRegression(
        **CLASSIFIER_CONFIG['LogisticRegression']['params'],
        random_state=RANDOM_SEED
    ))
])

# Entraîner le pipeline sur l'ensemble des données fusionnées
# pour obtenir un ensemble "final" de coefficients
model_l2_pipe.fit(X_combined, y_combined)

# Extraire le modèle entraîné (l'étape 'model') du pipeline
model_l2 = model_l2_pipe.named_steps['model']

# Extraire les noms des attributs
feature_names = X_combined.columns

coefficients = model_l2.coef_[0]

# Créer un DataFrame pour les coefficients
coeff_df = pd.DataFrame({
    'feature': feature_names,
    'coefficient': coefficients
})

# Trier par importance (valeur absolue)
coeff_df['abs_importance'] = np.abs(coeff_df['coefficient'])
coeff_df = coeff_df.sort_values(by='abs_importance', ascending=False)

print("\nTop 20 des attributs (basé sur les coefficients L1) :")
print(coeff_df.head(20))

# Graphique (Top 5)
top_20_coeffs = coeff_df.head(20).sort_values(by='abs_importance', ascending=True)
plt.figure(figsize=(10, 8))
plt.barh(top_20_coeffs['feature'], top_20_coeffs['coefficient'])
plt.title('Importance des Attributs (Coefficients Régression Logistique L2)')
plt.xlabel('Poids du Coefficient (les + importants sont loin de 0)')
plt.tight_layout()
plt.show()
# 

# --- 2. Importance basée sur la Permutation (Modèle L2) ---
print("\n  ... 2/2 Calcul de l'importance par permutation (plus long)...")

# n_repeats=10 est plus rapide que 30 pour un test
perm_imp = permutation_importance(
    model_l2_pipe, 
    X_combined, 
    y_combined, 
    scoring='balanced_accuracy', 
    n_repeats=10, 
    random_state=RANDOM_SEED, 
    n_jobs=-1
)

# Créer un DataFrame pour les scores de permutation
perm_df = pd.DataFrame({
    'feature': feature_names,
    'importance_mean': perm_imp.importances_mean
}).sort_values(by='importance_mean', ascending=False)

print("\nTop 20 des attributs (basé sur la Permutation) :")
print(perm_df.head(20))

# Graphique (Top 20)
top_20_perm = perm_df.head(20).sort_values(by='importance_mean', ascending=True)
plt.figure(figsize=(10, 8))
plt.barh(top_20_perm['feature'], top_20_perm['importance_mean'])
plt.title('Importance des Attributs par Permutation (sur Régression Logistique)')
plt.xlabel('Baisse moyenne de la "Balanced Accuracy"')
plt.tight_layout()
plt.show()


## Bloc de Type 3 — Améliorer nos résultats: Modèles Avancés et Évaluation Robuste

**: Évaluation Avancée avec LOSO (Leave-One-Subject-Out)**
La validation `StratifiedGroupKFold` (utilisée dans les blocs 1 et 2) est rapide et robuste. Cependant, la méthode "gold standard" pour l'EEG est la **LOSO**. Elle teste la capacité du modèle à généraliser à un *nouveau sujet* qui n'a jamais été vu pendant l'entraînement.

**Des idées :**
1.  **Créez un nouveau dictionnaire `models_advanced`**.
2.  Incluez-y les modèles de base (`LogisticRegression` L2, `RandomForest` ...).
3.  **Ajoutez de nouveaux modèles** pour comparer :
    * Un pipeline pour `LinearDiscriminantAnalysis` (LDA) (une baseline rapide).
    * Un pipeline pour `LogisticRegression` avec `penalty='l1'` (Lasso) et un `solver='liblinear'`.
4.  **Créez le validateur LOSO :** Importez `LeaveOneGroupOut` de `sklearn.model_selection` et instanciez-le : `loso_cv = LeaveOneGroupOut()`.
5.  **Évaluez :** Appelez votre fonction `evaluate_models` en lui passant votre nouveau `models_advanced`, les données `X_comb`, et `cv=loso_cv`.
6.  **Analysez :** Comparez les scores LOSO aux scores `StratifiedGroupKFold` du Bloc 1. (Attendez-vous à ce qu'ils soient plus bas ! C'est normal, car la tâche de généralisation inter-sujet est beaucoup plus difficile).

**Partie 3 : Analyse de l'Importance des Attributs**
Maintenant, nous voulons savoir *quels* attributs sont les plus importants. Nous allons comparer les résultats de deux types de modèles différents. Nous pouvons faire ça avec les modeles entrainés sur ERP oú fréquence seulement. 

**Hints :**
1.  **Modèle 1 : Régression Logistique (L1 - Lasso)**
    * Entraînez le pipeline `LogisticRegression_L1` (que vous venez de définir) sur *l'ensemble* des données `X_comb`, `y_comb`.
    * Extrayez les coefficients (`.coef_`) du modèle.
    * Créez un DataFrame et un graphique à barres (`plt.barh`) pour visualiser les poids. La pénalité L1 force les attributs inutiles à zéro, c'est donc une excellente méthode de *sélection d'attributs*.

2.  **Modèle 2 : Random Forest (Importance Gini)**
    * Entraînez le pipeline `RandomForest` (de `models_advanced`) sur *l'ensemble* des données.
    * Extrayez les importances (`.feature_importances_`).
    * Créez un graphique à barres pour visualiser "l'importance Gini".

**Partie 4 : (Optionnel) Importance par Permutation**
L'importance par permutation est une méthode robuste, agnostique au modèle, qui évalue l'importance d'un attribut en mesurant la baisse de performance lorsque cet attribut est mélangé aléatoirement.

## 3. Allez plus loin (Optionel):

### Bloc de Type 3 (Optionnel) — Optimisation des hyperparamètres (Grid Search)

Nous avons utilisé des modèles avec des paramètres par défaut. Une étape cruciale de l'apprentissage automatique consiste à trouver la **meilleure combinaison** d'hyperparamètres pour un problème donné. Nous allons utiliser `GridSearchCV` (recherche par grille) pour optimiser notre Régression Logistique.

Nous allons rechercher le paramètre `C` (l'inverse de la force de régularisation) et comparer les pénalités `L1 (Lasso)` et `L2 (Ridge)`.

* La pénalité **L1 (Lasso)** peut réduire les coefficients des attributs inutiles à zéro, effectuant ainsi une sélection automatique et donnant un modèle "parcimonieux" (sparse).
* La pénalité **L2 (Ridge)** répartit la réduction des poids plus uniformément et est numériquement stable lorsque de nombreux attributs sont corrélés.

**Vos tâches :**
1.  Importez `GridSearchCV` depuis `sklearn.model_selection`.
2.  Créez un `Pipeline` de base contenant uniquement le `StandardScaler` et une `LogisticRegression` (ex: `pipe_lr_grid = Pipeline([('scaler', StandardScaler()), ('model', LogisticRegression(max_iter=1000, class_weight='balanced'))])`).
3.  Créez un dictionnaire `param_grid` pour tester différentes options. **Attention :** pour tester des paramètres d'un pipeline, vous devez utiliser la syntaxe `etape__parametre`.
    * *Exemple de grid :*
        ```python
        param_grid = {
            'model__penalty': ['l1', 'l2'],
            'model__C': [0.01, 0.1, 1, 10],
            'model__solver': ['liblinear'] # 'liblinear' est un bon solveur qui gère L1 et L2
        }
        ```
4.  Instanciez `GridSearchCV`. Passez-lui le pipeline (`pipe_lr_grid`), le `param_grid`, et `cv=make_cv()` pour utiliser notre validation croisée groupée. N'oubliez pas `scoring='balanced_accuracy'`.
5.  Appelez `.fit(X_comb, y_comb, groups=groups_comb)` sur l'objet GridSearchCV.
6.  Affichez les meilleurs paramètres trouvés (`.best_params_`) et le meilleur score (`.best_score_`). Le L1 ou le L2 est-il meilleur pour ce problème ?

### Bloc de Type 3 (Optionnel) — Sélection d'attributs (Feature Selection)

Nous avons vu avec le L1 (Lasso) que certains attributs sont mis à zéro. Et si nous entraînions un modèle *uniquement* sur les N meilleurs attributs ? L'utilisation d'un sous-ensemble d'attributs peut réduire le surapprentissage (overfitting), accélérer l'entraînement et améliorer l'interprétabilité.

Nous utiliserons `SelectKBest` avec un test statistique `f_classif` (ANOVA F-value) pour sélectionner les `k` attributs les plus corrélés avec la cible.

**Vos tâches :**
1.  Importez `SelectKBest` et `f_classif` depuis `sklearn.feature_selection`.
2.  Créez un nouveau `Pipeline` qui inclut cette étape de sélection. **L'ordre est crucial :**
    1.  `StandardScaler`
    2.  `SelectKBest(f_classif, k=15)` (Essayez avec `k=15` attributs pour commencer)
    3.  Votre modèle (ex: `LogisticRegression(...)` ou `RandomForestClassifier(...)`)
3.  Créez un nouveau dictionnaire `models_selection` contenant ce pipeline (ex: `{'LR_L2_k15': pipeline_k15}`).
4.  Appelez `evaluate_models` (avec `cv=make_cv()` ou `loso_cv`) pour évaluer ce nouveau pipeline sur les données fusionnées (`X_comb`, `y_comb`, `groups_comb`).
5.  Le score s'améliore-t-il en ne gardant que les 15 meilleurs attributs ? Comparez ce score à celui du modèle L1 (Lasso), qui effectue sa propre sélection d'attributs intégrée.

### Bloc de Type 3 (Optionnel) — Test de permutation (Validation du modèle)

Notre meilleur modèle a obtenu un score (disons 65% de balanced accuracy). Est-ce significativement mieux que le hasard (50%) ? Le test de permutation est la méthode "gold standard" pour répondre à cette question.

Il fonctionne en ré-entraînant le modèle des centaines de fois, mais en **mélangeant aléatoirement les étiquettes (y)** à chaque fois. Cela crée une distribution de scores "dus au hasard". Nous comparons ensuite notre score réel à cette distribution.

**Vos tâches :**
1.  Importez `permutation_test_score` depuis `sklearn.model_selection`.
2.  Choisissez votre **meilleur** pipeline (ex: `models_advanced['LogisticRegression_L1']`) et le **meilleur** jeu de données (`X_comb`, `y_comb`, `groups_comb`).
3.  Choisissez votre **CV** (le `loso_cv` est le plus robuste, mais `make_cv()` est plus rapide).
4.  Appelez la fonction `permutation_test_score`. **Attention, cela peut être très long !** Commencez avec `n_permutations=100`.
    * `score, perm_scores, pvalue = permutation_test_score(`
    * `  model, X_comb, y_comb, groups=groups_comb, cv=loso_cv,`
    * `  scoring='balanced_accuracy', n_permutations=100, n_jobs=-1`
    * `)`
5.  Affichez le `pvalue`. Si `pvalue < 0.05`, cela signifie que votre score de modèle est **statistiquement significatif** et n'est pas dû au hasard.
6.  Affichez le `score` (votre score réel) et comparez-le à `np.mean(perm_scores)` (le score moyen du hasard).
7.  (Bonus) Utilisez `plt.hist` pour tracer l'histogramme des `perm_scores` et ajoutez une ligne verticale rouge (`plt.axvline`) pour votre `score` réel afin de visualiser à quel point il est "exceptionnel".